# Indonesia Resource-Adjusted GVA Map

Generates a self-contained interactive HTML map with Folium + Leaflet:
- Provincial GVA (ADHK 2010 prices)
- Resource depletion costs (SISNERLING 2020-2024)
- Resource-adjusted regional output (NDRP)
- On-click panel: sparkline, sector bar chart, waterfall contribution chart

Run cells sequentially.

In [51]:
import pandas as pd
import json
import math
import copy
import urllib.request
from pathlib import Path

import folium
from folium import GeoJson, GeoJsonTooltip, GeoJsonPopup
import branca.colormap as cm

# 1.  DEPLETION TIME SERIES  (billion Rp, SISNERLING App. 65-69)

In [52]:
DEPLETION_SERIES = {
    2020: dict(coal=56624,  oil=95012,  gas=75884,  copper=10871, timber=33527,
               gold=3478,   silver=155, tin=3417,   nickel=2044,  bauxite=796),
    2021: dict(coal=88615,  oil=89111,  gas=94345,  copper=28116, timber=38088,
               gold=7951,   silver=195, tin=3599,   nickel=4342,  bauxite=1064),
    2022: dict(coal=216088, oil=110762, gas=115554, copper=66744, timber=40555,
               gold=11260,  silver=144, tin=8617,   nickel=6415,  bauxite=1676),
    2023: dict(coal=333305, oil=105018, gas=125788, copper=68937, timber=142513,
               gold=10683,  silver=38,  tin=17415,  nickel=11822, bauxite=897),
    2024: dict(coal=306797, oil=89463,  gas=123062, copper=81347, timber=72169,
               gold=7393,   silver=98,  tin=13741,  nickel=12342, bauxite=610),
}

# 2.  PROVINCIAL COMMODITY SHARES
Each dict maps PROVINCE_KEY -> fraction of national depletion

Basis: share of sector B mining PDRB + ESDM/SKK Migas production geography

In [53]:
PROV_COMMODITY_SHARES = {
    "coal":    {"KALIMANTAN TIMUR": 0.40, "KALIMANTAN SELATAN": 0.25,
                "SUMATERA SELATAN": 0.15, "KALIMANTAN TENGAH": 0.08,
                "JAMBI": 0.05},
    "oil":     {"KALIMANTAN TIMUR": 0.35, "RIAU": 0.30,
                "PAPUA BARAT": 0.12, "JAWA TIMUR": 0.10},
    "gas":     {"KALIMANTAN TIMUR": 0.45, "RIAU": 0.20,
                "PAPUA BARAT": 0.15, "ACEH": 0.08},
    "copper":  {"PAPUA TENGAH": 0.85, "NUSA TENGGARA BARAT": 0.15},
    "timber":  {"KALIMANTAN TENGAH": 0.25, "KALIMANTAN TIMUR": 0.20,
                "PAPUA": 0.20, "KALIMANTAN BARAT": 0.15},
    "gold":    {"PAPUA TENGAH": 0.60, "NUSA TENGGARA BARAT": 0.20,
                "SULAWESI TENGGARA": 0.10},
    "silver":  {"PAPUA TENGAH": 0.55, "NUSA TENGGARA BARAT": 0.25,
                "SULAWESI TENGGARA": 0.10},
    "tin":     {"KEPULAUAN BANGKA BELITUNG": 0.90, "KEPULAUAN RIAU": 0.10},
    "nickel":  {"MALUKU UTARA": 0.35, "SULAWESI TENGAH": 0.30,
                "SULAWESI TENGGARA": 0.25},
    "bauxite": {"KALIMANTAN BARAT": 0.95},
}

COMMODITY_LABELS = {
    "coal": "Coal", "oil": "Crude Oil", "gas": "Natural Gas",
    "copper": "Copper", "timber": "Timber", "gold": "Gold",
    "silver": "Silver", "tin": "Tin", "nickel": "Nickel", "bauxite": "Bauxite",
}

# 3.  PROVINCIAL GVA DATA
GVA in trillion Rp (ADHK, constant 2010 prices)
YoY growth rates in percent (same quarter, prior year)
Sector time series for on-click charts

In [54]:
# -- Path to 02_intermediate_data -- edit INTER if your folder moves ----------
INTER = Path(r'C:\\Users\\Admin\\OneDrive\\Desktop\\Personal Projects\\Indonesia GVA\\02_intermediate_data')
if not (INTER / '03_01_provincial_gva_quarterly.csv').exists():
    _nb_dir = Path(globals().get('__vsc_ipynb_file__', '.')).resolve().parent
    for _c in [_nb_dir, _nb_dir.parent / '02_intermediate_data', Path.cwd()]:
        if (_c / '03_01_provincial_gva_quarterly.csv').exists():
            INTER = _c
            break
    else:
        raise FileNotFoundError(
            'Cannot find 03_01_provincial_gva_quarterly.csv.\n'
            'Update the INTER path at the top of this cell.'
        )
print(f'Data directory: {INTER}')

# -- Load quarterly GVA outputs from 03_01 ------------------------------------
gva_qtr = pd.read_csv(INTER / '03_01_provincial_gva_quarterly.csv')

adhk = gva_qtr[gva_qtr['sector_code'] != 'PDRB'].copy()

# -- Sector-level YoY (same quarter, prior year) ------------------------------
adhk = adhk.sort_values(['provinsi', 'sector_code', 'year', 'quarter'])
adhk['gva_yoy'] = (
    adhk.groupby(['provinsi', 'sector_code', 'quarter'])['gva_sector']
    .pct_change(fill_method=None) * 100
)

# -- Provincial ADHK totals per quarter + YoY ---------------------------------
prov_total = (
    adhk.groupby(['provinsi', 'year', 'quarter', 'period'])['gva_sector']
    .sum().reset_index()
    .sort_values(['provinsi', 'year', 'quarter'])
)
prov_total['gva_yoy'] = (
    prov_total.groupby(['provinsi', 'quarter'])['gva_sector']
    .pct_change(fill_method=None) * 100
)
prov_total['gva_trillion'] = prov_total['gva_sector'] / 1000

# -- Sector shares ------------------------------------------------------------
sector_shares = adhk.merge(
    prov_total[['provinsi', 'period', 'gva_sector']].rename(columns={'gva_sector': 'prov_total'}),
    on=['provinsi', 'period']
)
sector_shares['share_pct'] = sector_shares['gva_sector'] / sector_shares['prov_total'] * 100

def top_sectors(df, period, n=6):
    sub = df[df['period'] == period].sort_values('share_pct', ascending=False)
    top = sub.head(n)[['sector_short', 'share_pct']]
    return dict(zip(top['sector_short'], top['share_pct'].round(1)))

# -- Build PROVINCE_DATA ------------------------------------------------------
PROVINCE_DATA = {}
periods_available = sorted(adhk['period'].unique())

for prov in sorted(adhk['provinsi'].unique()):
    key = prov.upper()
    pt  = prov_total[prov_total['provinsi'] == prov].set_index('period')
    ss  = sector_shares[sector_shares['provinsi'] == prov]
    latest_full_period = next(
        (p for p in reversed(periods_available) if p in pt.index), None
    )
    sec_df = adhk[adhk['provinsi'] == prov][
        ['period', 'sector_short', 'gva_sector', 'gva_yoy']
    ].dropna(subset=['sector_short'])
    sector_ts = {}
    for _, row in sec_df.iterrows():
        sec = row['sector_short']
        sector_ts.setdefault(sec, {})[row['period']] = {
            'gva': round(row['gva_sector'] / 1000, 4),
            'yoy': round(row['gva_yoy'], 3) if not pd.isna(row['gva_yoy']) else None,
        }
    PROVINCE_DATA[key] = {
        'gva':       pt['gva_trillion'].dropna().to_dict(),
        'yoy':       pt['gva_yoy'].dropna().to_dict(),
        'sectors':   top_sectors(ss, latest_full_period) if latest_full_period else {},
        'sector_ts': sector_ts,
    }

print(f"PROVINCE_DATA built: {len(PROVINCE_DATA)} provinces")
print(f"Periods: {periods_available[:4]} ... {periods_available[-4:]}")
print(f"Sample -- DKI JAKARTA 2024Q4:")
print(f"  GVA: {PROVINCE_DATA['DKI JAKARTA']['gva'].get('2024Q4', 'N/A'):.1f}T")
print(f"  YoY: {PROVINCE_DATA['DKI JAKARTA']['yoy'].get('2024Q4', 'N/A'):.2f}%")
print(f"  Sector TS keys: {list(PROVINCE_DATA['DKI JAKARTA']['sector_ts'].keys())[:3]}")

Data directory: C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data
PROVINCE_DATA built: 38 provinces
Periods: ['2020Q1', '2020Q2', '2020Q3', '2020Q4'] ... ['2025Q2', '2025Q3', '2025Q4', '2026Q1']
Sample -- DKI JAKARTA 2024Q4:
  GVA: 546.8T
  YoY: 5.01%
  Sector TS keys: ['Pertanian', 'Pertambangan', 'Industri Pengolahan']


# 4.  HELPER FUNCTIONS

In [55]:
def calc_depletion(province_key: str, year: int) -> dict:
    dep = DEPLETION_SERIES[year]
    total = 0.0
    breakdown = {}
    for commodity, prov_shares in PROV_COMMODITY_SHARES.items():
        share = prov_shares.get(province_key, 0.0)
        val = dep.get(commodity, 0) * share
        if val > 0:
            breakdown[commodity] = val
            total += val
    return {"total": total, "breakdown": breakdown}


def fmt_rp(billion_rp: float) -> str:
    if not billion_rp or billion_rp < 1:
        return "< Rp1B"
    if billion_rp >= 1000:
        return f"Rp{billion_rp/1000:.1f}T"
    return f"Rp{billion_rp:,.0f}B"


def normalize_name(raw: str) -> str:
    name = raw.upper().strip()
    for prefix in ("PROVINSI ", "PROVINCE OF ", "PROV. "):
        name = name.replace(prefix, "")
    aliases = {
        "KEPULAUAN BANGKA-BELITUNG":  "KEP. BANGKA BELITUNG",
        "KEPULAUAN BANGKA BELITUNG":  "KEP. BANGKA BELITUNG",
        "KEPULAUAN RIAU":             "KEP. RIAU",
        "BANGKA BELITUNG":            "KEP. BANGKA BELITUNG",
        "BANGKA-BELITUNG":            "KEP. BANGKA BELITUNG",
        "JAKARTA RAYA":               "DKI JAKARTA",
        "JAKARTA":                    "DKI JAKARTA",
        "YOGYAKARTA":                 "DI YOGYAKARTA",
        "D.I. YOGYAKARTA":            "DI YOGYAKARTA",
        "WEST JAVA":                  "JAWA BARAT",
        "CENTRAL JAVA":               "JAWA TENGAH",
        "EAST JAVA":                  "JAWA TIMUR",
        "WEST SUMATRA":               "SUMATERA BARAT",
        "NORTH SUMATRA":              "SUMATERA UTARA",
        "SOUTH SUMATRA":              "SUMATERA SELATAN",
        "EAST KALIMANTAN":            "KALIMANTAN TIMUR",
        "WEST KALIMANTAN":            "KALIMANTAN BARAT",
        "CENTRAL KALIMANTAN":         "KALIMANTAN TENGAH",
        "SOUTH KALIMANTAN":           "KALIMANTAN SELATAN",
        "NORTH KALIMANTAN":           "KALIMANTAN UTARA",
        "NORTH SULAWESI":             "SULAWESI UTARA",
        "CENTRAL SULAWESI":           "SULAWESI TENGAH",
        "SOUTH SULAWESI":             "SULAWESI SELATAN",
        "SOUTHEAST SULAWESI":         "SULAWESI TENGGARA",
        "WEST SULAWESI":              "SULAWESI BARAT",
        "NORTH MALUKU":               "MALUKU UTARA",
        "WEST PAPUA":                 "PAPUA BARAT",
        "NUSA TENGGARA WEST":         "NUSA TENGGARA BARAT",
        "NUSA TENGGARA EAST":         "NUSA TENGGARA TIMUR",
    }
    return aliases.get(name, name)


def _valid(features, key):
    return [
        v for f in features
        if (v := f["properties"].get(key)) is not None
        and isinstance(v, (int, float))
        and not math.isnan(v)
    ]


def _range(vals, fallback_min=0, fallback_max=1):
    lo = min(vals, default=fallback_min)
    hi = max(vals, default=fallback_max)
    return lo, hi if hi > lo else lo + 1


def enrich_geojson(geojson: dict, province_data: dict, year: int, quarter: str) -> dict:
    """Enrich GeoJSON features with GVA, depletion, and sector data."""
    geojson  = copy.deepcopy(geojson)
    period   = f"{year}{quarter}"
    dep_year = year if year in DEPLETION_SERIES else max(k for k in DEPLETION_SERIES if k <= year)
    for feat in geojson["features"]:
        props = feat["properties"]
        raw_name = (props.get("state") or props.get("Propinsi") or
                    props.get("name") or props.get("NAME_1") or "")
        key   = normalize_name(raw_name)
        pdata = province_data.get(key)
        dep   = calc_depletion(key, dep_year)
        props["province_key"] = key
        if pdata:
            raw_gva = pdata["gva"].get(period)
            yoy     = pdata["yoy"].get(period)
            dep_t   = dep["total"] / 4 / 1000
            adj_gva = (raw_gva - dep_t) if raw_gva else None
            haircut = (dep_t / raw_gva * 100) if (raw_gva and raw_gva > 0) else 0
            top3    = [
                (s, v) for s, v in
                sorted(pdata["sectors"].items(), key=lambda x: -x[1])
                if s not in ("Lainnya", "Others")
            ][:3]
            props.update({
                "raw_gva":       f"Rp {raw_gva:.1f}T"  if raw_gva else "N/A",
                "yoy":           f"{yoy:+.1f}%"         if yoy is not None else "N/A",
                "dep_total_str": fmt_rp(dep["total"] / 4),
                "adj_gva":       f"Rp {adj_gva:.1f}T"  if adj_gva else "N/A",
                "haircut":       f"{haircut:.1f}%",
                "top3_sectors":  " | ".join(f"{s}: {v:.0f}%" for s, v in top3),
                "dep_breakdown": (" + ".join(
                    f"{COMMODITY_LABELS[c]}: {fmt_rp(v / 4)}"
                    for c, v in sorted(dep["breakdown"].items(), key=lambda x: -x[1])
                ) if dep["breakdown"] else "None"),
                "sectors":       pdata["sectors"],
                "sector_ts":     pdata.get("sector_ts", {}),
                "val_yoy":       yoy     if yoy is not None else float("nan"),
                "val_gva":       raw_gva if raw_gva         else float("nan"),
                "val_dep":       dep_t,
                "val_adj_gva":   adj_gva if adj_gva         else float("nan"),
                "val_haircut":   haircut,
            })
        else:
            props.update({
                "raw_gva": "N/A", "yoy": "N/A", "dep_total_str": "N/A",
                "adj_gva": "N/A", "haircut": "0%",
                "top3_sectors": "N/A", "dep_breakdown": "No data",
                "sectors": {}, "sector_ts": {},
                "val_yoy": float("nan"), "val_gva": float("nan"),
                "val_dep": 0, "val_adj_gva": float("nan"), "val_haircut": 0,
            })
    return geojson

# 5.  GEOJSON LOADING

In [56]:
#
#  Option A: auto-download from GitHub (runs automatically below)
#    https://github.com/superpikar/indonesia-geojson/blob/master/indonesia.geojson
#
#  Option B: GADM higher quality
#    https://gadm.org/download_country.html -> Indonesia -> GeoJSON level 1
#    Save as: indonesia_provinces.geojson
#
GEOJSON_PATH = Path("indonesia_provinces.geojson")
GEOJSON_URLS = [
    "https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia.geojson",
    "https://github.com/superpikar/indonesia-geojson/raw/master/indonesia.geojson",
]

def load_geojson() -> dict:
    if GEOJSON_PATH.exists():
        print(f"Using GeoJSON: {GEOJSON_PATH}")
        with open(GEOJSON_PATH) as f:
            return json.load(f)
    print("GeoJSON not found locally -- attempting download...")
    for url in GEOJSON_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Python/3"})
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = resp.read()
            GEOJSON_PATH.write_bytes(data)
            print(f"  Downloaded from {url}")
            return json.loads(data)
        except Exception as e:
            print(f"  Failed ({url}): {e}")
    raise FileNotFoundError(
        "Could not download GeoJSON automatically.\n"
        "Download manually from:\n"
        "  https://github.com/superpikar/indonesia-geojson/blob/master/indonesia.geojson\n"
        f"Save as: {GEOJSON_PATH.resolve()}"
    )

# 6.  NAME ALIGNMENT DIAGNOSTIC
Run once after loading data to verify GeoJSON <-> CSV province name matching.
Names in the first block appear grey on the map.

In [57]:
geo_raw   = load_geojson()
geo_names = set()
for feat in geo_raw["features"]:
    p   = feat["properties"]
    raw = (p.get("state") or p.get("Propinsi") or p.get("name") or p.get("NAME_1") or "")
    geo_names.add(normalize_name(raw))

csv_names = set(PROVINCE_DATA.keys())

print("=== In GeoJSON but NOT in CSV (grey on map) ===")
for n in sorted(geo_names - csv_names): print(" ", n)

print("\n=== In CSV but NOT in GeoJSON (never drawn) ===")
for n in sorted(csv_names - geo_names): print(" ", n)

print(f"\n=== Matched: {len(geo_names & csv_names)} provinces ===")

Using GeoJSON: indonesia_provinces.geojson
=== In GeoJSON but NOT in CSV (grey on map) ===

=== In CSV but NOT in GeoJSON (never drawn) ===
  PAPUA BARAT DAYA
  PAPUA PEGUNUNGAN
  PAPUA SELATAN
  PAPUA TENGAH

=== Matched: 34 provinces ===


# 7.  MAP BUILDER

In [58]:
def build_map(
    province_data: dict,
    years:    list = [2024],
    quarters: list = ["Q1", "Q2", "Q3", "Q4"],
    output:   str  = "indonesia_gva_depletion_map.html",
) -> folium.Map:

    # -- Build one enriched GeoJSON per period --------------------------------
    all_periods = [f"{y}{q}" for y in sorted(years) for q in quarters]
    geo_by_period = {}
    for period in all_periods:
        if period not in set(periods_available):   # ← add this
            continue
        y, q = int(period[:4]), period[4:]
        enriched = enrich_geojson(load_geojson(), province_data, y, q)
        geo_by_period[period] = enriched
        print(f"  Enriched {period}")

    all_periods = list(geo_by_period.keys())  

    # -- Compute global colormap bounds across all periods --------------------
    all_feats = [f for p in all_periods for f in geo_by_period[p]["features"]]

    yoy_lo, yoy_hi = -2, 10
    gva_lo, gva_hi = _range(_valid(all_feats, "val_gva"),     0, 700)
    dep_lo, dep_hi = _range(_valid(all_feats, "val_dep"),     0, 250)
    hc_lo,  hc_hi  = _range(_valid(all_feats, "val_haircut"), 0, 80)

    # -- Base map -------------------------------------------------------------
    m = folium.Map(
        location=[-2.5, 118], zoom_start=5,
        tiles="CartoDB dark_matter", control_scale=True,
    )

    # -- Serialise all period data for JS ------------------------------------
    periods_json = json.dumps(all_periods)
    geo_data_js  = json.dumps({p: geo_by_period[p] for p in all_periods})

    # -- Pre-compute HTML fragments ------------------------------------------
    _layer_buttons = ''.join(
        f'<button class="layer-btn" data-layer="{k}" onclick="setLayer(\'{k}\')"'
        f' style="display:block;width:100%;margin-bottom:4px;padding:4px 8px;'
        f'background:#0d1a2d;color:#f0e8d8;border:1px solid #2a3a5a;'
        f'cursor:pointer;font-family:\'Courier New\',monospace;font-size:10px;">'
        f'{v}</button>'
        for k, v in [("yoy",     "YoY Growth"),
                     ("gva",     "GVA Level"),
                     ("dep",     "Depletion Cost"),
                     ("haircut", "Depletion % GVA")]
    )
    _slider_max   = len(all_periods) - 1
    _slider_start = all_periods[0]
    _slider_end   = all_periods[-1]

    m.get_root().html.add_child(folium.Element(f"""
    <script>
    var ALL_PERIODS   = {periods_json};
    var GEO_BY_PERIOD = {geo_data_js};

    var YOY_LO = {yoy_lo}, YOY_HI = {yoy_hi};
    var GVA_LO = {gva_lo}, GVA_HI = {gva_hi};
    var DEP_LO = {dep_lo}, DEP_HI = {dep_hi};
    var HC_LO  = {hc_lo},  HC_HI  = {hc_hi};

    function lerp(a, b, t) {{ return a + (b - a) * t; }}
    function clamp(v, lo, hi) {{ return Math.max(lo, Math.min(hi, v)); }}

    function colorScale(v, lo, hi, c0, cmid, c1) {{
        if (v === null || v === undefined || isNaN(v)) return "#4a4a4a";
        var t = (clamp(v, lo, hi) - lo) / (hi - lo);
        var r, g, b;
        if (t < 0.5) {{
            var s = t * 2;
            r = Math.round(lerp(c0[0], cmid[0], s));
            g = Math.round(lerp(c0[1], cmid[1], s));
            b = Math.round(lerp(c0[2], cmid[2], s));
        }} else {{
            var s = (t - 0.5) * 2;
            r = Math.round(lerp(cmid[0], c1[0], s));
            g = Math.round(lerp(cmid[1], c1[1], s));
            b = Math.round(lerp(cmid[2], c1[2], s));
        }}
        return "rgb(" + r + "," + g + "," + b + ")";
    }}

    function colorYoy(v) {{
        return colorScale(v, YOY_LO, YOY_HI,
            [192,57,43], [245,240,232], [26,107,107]);
    }}
    function colorGva(v) {{
        if (!v || v === 0) return "#4a4a4a";
        return colorScale(v, GVA_LO, GVA_HI,
            [239,246,255], [59,130,246], [30,58,95]);
    }}
    function colorDep(v) {{
        if (!v) return "#4a4a4a";
        return colorScale(v, DEP_LO, DEP_HI,
            [254,249,240], [192,57,43], [123,26,26]);
    }}
    function colorHaircut(v) {{
        if (!v) return "#4a4a4a";
        return colorScale(v, HC_LO, HC_HI,
            [254,249,240], [139,69,19], [74,26,0]);
    }}

    function colorGva(v) {{
        if (!v || v === 0) return "#4a4a4a";
        return colorScale(v, GVA_LO, GVA_HI, [239,246,255], [59,130,246], [30,58,95]);
    }}
    function colorDep(v) {{
        if (!v) return "#4a4a4a";
        return colorScale(v, DEP_LO, DEP_HI, [254,249,240], [192,57,43], [123,26,26]);
    }}
    function colorHaircut(v) {{
        if (!v) return "#4a4a4a";
        return colorScale(v, HC_LO, HC_HI, [254,249,240], [139,69,19], [74,26,0]);
    }}

    var SECTOR_COLOURS = {{
        "Pertanian":"#2D6A4F","Pertambangan":"#B5838D","Industri Pengolahan":"#457B9D",
        "Listrik & Gas":"#E9C46A","Air & Sampah":"#84A98C","Konstruksi":"#E76F51",
        "Perdagangan":"#264653","Transportasi":"#6D6875","Akomodasi":"#F4A261",
        "Infokom":"#2A9D8F","Keuangan":"#E63946","Real Estate":"#A8DADC",
        "Jasa Perusahaan":"#9B2226","Adm. Pemerintahan":"#AE2012",
        "Pendidikan":"#0A9396","Kesehatan":"#94D2BD","Jasa Lainnya":"#CA6702"
    }};

    var activeLayer   = "yoy";
    var geoLayer      = null;
    var currentPeriod = ALL_PERIODS[ALL_PERIODS.length - 1];

    function getColor(props) {{
        if (activeLayer === "yoy")     return colorYoy(props.val_yoy);
        if (activeLayer === "gva")     return colorGva(props.val_gva);
        if (activeLayer === "dep")     return colorDep(props.val_dep);
        if (activeLayer === "haircut") return colorHaircut(props.val_haircut);
        return "#4a4a4a";
    }}

    function redraw(period) {{
        currentPeriod = period;
        var _map = Object.values(window).find(function(v) {{ return v instanceof L.Map; }});
        if (!_map) return;
        if (geoLayer) {{ _map.removeLayer(geoLayer); }}
        var geo = GEO_BY_PERIOD[period];
        geoLayer = L.geoJson(geo, {{
            style: function(f) {{
                return {{
                    fillColor: getColor(f.properties),
                    color: "#6a7d60", weight: 0.8, fillOpacity: 0.82
                }};
            }},
            onEachFeature: function(f, layer) {{
                var p = f.properties;
                layer.bindTooltip(
                    "<b style='color:#c9a84c'>" + (p.province_key || "") + "</b><br>" +
                    "GVA (ADHK): " + (p.raw_gva      || "N/A") + "<br>" +
                    "YoY Growth: " + (p.yoy           || "N/A") + "<br>" +
                    "Depletion: "  + (p.dep_total_str || "N/A") + "<br>" +
                    "Haircut: "    + (p.haircut        || "N/A") + "<br>" +
                    "Adj. NDRP: "  + (p.adj_gva        || "N/A") + "<br>" +
                    "<span style='color:#7090b0;font-size:9px'>click for full analysis</span>",
                    {{sticky: true, className: "custom-tip"}}
                );
                layer.on("mouseover", function() {{
                    layer.setStyle({{weight: 2.5, color: "#b8860b", fillOpacity: 0.95}});
                }});
                layer.on("mouseout", function() {{
                    geoLayer.resetStyle(layer);
                }});
                layer.on("click", function() {{
                    openPanel(p, currentPeriod);
                }});
            }}
        }}).addTo(_map);
        document.getElementById("period-label").innerText = period;
    }}

    function setLayer(name) {{
        activeLayer = name;
        document.querySelectorAll(".layer-btn").forEach(function(b) {{
            b.style.borderColor = b.dataset.layer === name ? "#b8860b" : "#2a3a5a";
        }});
        redraw(currentPeriod);
    }}

    function svgEl(tag, attrs, text) {{
        var el = document.createElementNS("http://www.w3.org/2000/svg", tag);
        for (var k in attrs) el.setAttribute(k, attrs[k]);
        if (text !== undefined) el.textContent = text;
        return el;
    }}

    function drawSparkline(provinceKey) {{
        var svg = document.getElementById("sparkline");
        while (svg.firstChild) svg.removeChild(svg.firstChild);
        var yoyMap = {{}};
        ALL_PERIODS.forEach(function(p) {{
            var geo = GEO_BY_PERIOD[p];
            if (!geo) return;
            var feat = geo.features.find(function(f) {{
                return f.properties.province_key === provinceKey;
            }});
            if (feat) {{
                var v = feat.properties.val_yoy;
                if (v != null && !isNaN(v)) yoyMap[p] = v;
            }}
        }});
        var periods = ALL_PERIODS.filter(function(p) {{ return yoyMap[p] != null; }});
        if (periods.length < 2) return;
        var vals = periods.map(function(p) {{ return yoyMap[p]; }});
        var W = 340, H = 90, padX = 6, padY = 18;
        var lo = Math.min.apply(null, vals), hi = Math.max.apply(null, vals);
        var rng = hi - lo || 1;
        var xStep = (W - padX*2) / (periods.length - 1);
        function cx(i) {{ return padX + i * xStep; }}
        function cy(v) {{ return H - padY - ((v - lo) / rng) * (H - padY*2); }}

        svg.appendChild(svgEl("line", {{x1:padX, y1:cy(0), x2:W-padX, y2:cy(0),
            stroke:"#2a3a5a", "stroke-width":"1", "stroke-dasharray":"3,3"}}));

        var pts = periods.map(function(p,i) {{ return [cx(i), cy(yoyMap[p])]; }});
        var d = "M"+pts[0][0]+","+pts[0][1];
        for (var i=1;i<pts.length;i++) d += " L"+pts[i][0]+","+pts[i][1];
        d += " L"+pts[pts.length-1][0]+","+(H-padY)+" L"+pts[0][0]+","+(H-padY)+" Z";
        svg.appendChild(svgEl("path", {{d:d, fill:"rgba(30,160,90,0.15)", stroke:"none"}}));

        var ln = "M"+pts[0][0]+","+pts[0][1];
        for (var i=1;i<pts.length;i++) ln += " L"+pts[i][0]+","+pts[i][1];
        svg.appendChild(svgEl("path", {{d:ln, fill:"none", stroke:"#1ea05a", "stroke-width":"1.8"}}));

        var special = [0, periods.length-1,
            vals.indexOf(Math.min.apply(null,vals)),
            vals.indexOf(Math.max.apply(null,vals))];
        special = special.filter(function(v,i,a){{return a.indexOf(v)===i;}});
        special.forEach(function(i) {{
            var v = vals[i], col = v >= 0 ? "#1ea05a" : "#c0392b";
            svg.appendChild(svgEl("circle", {{cx:cx(i), cy:cy(v), r:"3", fill:col}}));
            svg.appendChild(svgEl("text", {{
                x:cx(i), y: cy(v) > H/2 ? cy(v)-5 : cy(v)+11,
                "text-anchor":"middle", fill:col,
                "font-size":"8", "font-family":"Courier New"
            }}, (v>=0?"+":"")+v.toFixed(1)+"%"));
        }});
        periods.forEach(function(p,i) {{
            if (i % 2 !== 0) return;
            svg.appendChild(svgEl("text", {{
                x:cx(i), y:H-2, "text-anchor":"middle",
                fill:"#506080", "font-size":"7", "font-family":"Courier New"
            }}, p));
        }});
    }}

    function drawSectorBars(sectors) {{
        var svg = document.getElementById("sector-bars");
        while (svg.firstChild) svg.removeChild(svg.firstChild);
        var entries = Object.entries(sectors).sort(function(a,b){{return b[1]-a[1];}});
        if (!entries.length) return;
        var W = 340, barH = 18, gap = 5, padL = 120, padR = 48;
        var maxVal = entries[0][1];
        var totalH = entries.length * (barH + gap);
        svg.setAttribute("height", totalH);
        entries.forEach(function(e, i) {{
            var name = e[0], val = e[1];
            var barW = Math.max(2, (val / maxVal) * (W - padL - padR));
            var y = i * (barH + gap);
            var col = SECTOR_COLOURS[name] || "#888780";
            svg.appendChild(svgEl("rect", {{x:padL, y:y, width:barW, height:barH, fill:col, rx:"2"}}));
            svg.appendChild(svgEl("text", {{
                x:padL-4, y:y+barH*0.72, "text-anchor":"end",
                fill:"#c8bfb0", "font-size":"9", "font-family":"Courier New"
            }}, name.length > 16 ? name.slice(0,15)+"..." : name));
            svg.appendChild(svgEl("text", {{
                x:padL+barW+4, y:y+barH*0.72, "text-anchor":"start",
                fill:"#c9a84c", "font-size":"9", "font-family":"Courier New"
            }}, val.toFixed(1)+"%"));
        }});
    }}

    function drawWaterfall(sectorTs, period) {{
        var svg = document.getElementById("waterfall");
        while (svg.firstChild) svg.removeChild(svg.firstChild);
        if (!sectorTs || !period) return;
        var pidx = ALL_PERIODS.indexOf(period);
        if (pidx < 4) return;
        var priorPeriod = ALL_PERIODS[pidx - 4];
        var totalPrior = 0;
        Object.keys(sectorTs).forEach(function(s) {{
            var pr = sectorTs[s][priorPeriod];
            if (pr) totalPrior += pr.gva;
        }});
        if (!totalPrior) return;
        var contribs = [];
        Object.keys(sectorTs).forEach(function(s) {{
            var curr = sectorTs[s][period], prior = sectorTs[s][priorPeriod];
            if (curr && prior) {{
                contribs.push({{name:s, pp:(curr.gva-prior.gva)/totalPrior*100}});
            }}
        }});
        contribs.sort(function(a,b){{return a.pp-b.pp;}});
        var W = 340, barH = 16, gap = 4, padL = 120, padR = 48;
        var totalH = contribs.length * (barH + gap) + 14;
        svg.setAttribute("height", totalH);
        var maxAbs = Math.max.apply(null, contribs.map(function(c){{return Math.abs(c.pp);}}))||1;
        var halfW = (W - padL - padR) / 2;
        var zeroX = padL + halfW;
        svg.appendChild(svgEl("line", {{x1:zeroX, y1:0, x2:zeroX, y2:totalH-14,
            stroke:"#2a3a5a", "stroke-width":"1"}}));
        contribs.forEach(function(c, i) {{
            var y = i*(barH+gap);
            var barW = (Math.abs(c.pp)/maxAbs)*halfW;
            var x = c.pp >= 0 ? zeroX : zeroX - barW;
            var col = c.pp >= 0 ? (SECTOR_COLOURS[c.name]||"#1ea05a") : "#c0392b";
            svg.appendChild(svgEl("rect", {{x:x, y:y, width:barW, height:barH, fill:col, rx:"2", opacity:"0.85"}}));
            svg.appendChild(svgEl("text", {{
                x:padL-4, y:y+barH*0.72, "text-anchor":"end",
                fill:"#c8bfb0", "font-size":"9", "font-family":"Courier New"
            }}, c.name.length>16 ? c.name.slice(0,15)+"..." : c.name));
            var vx = c.pp>=0 ? x+barW+3 : x-3;
            svg.appendChild(svgEl("text", {{
                x:vx, y:y+barH*0.72,
                "text-anchor": c.pp>=0 ? "start" : "end",
                fill: c.pp>=0 ? "#c9a84c" : "#e07070",
                "font-size":"9", "font-family":"Courier New"
            }}, (c.pp>=0?"+":"")+c.pp.toFixed(2)+"pp"));
        }});
        svg.appendChild(svgEl("text", {{
            x:W/2, y:totalH-2, "text-anchor":"middle",
            fill:"#506080", "font-size":"8", "font-family":"Courier New"
        }}, priorPeriod+" -> "+period));
    }}

    function closePanel() {{
        document.getElementById("click-panel").style.display = "none";
    }}

    function openPanel(props, period) {{
        document.getElementById("panel-title").textContent  = props.province_key || "";
        document.getElementById("panel-period").textContent = period;
        var rows = [
            ["GVA (ADHK)",   props.raw_gva      || "N/A"],
            ["YoY Growth",   props.yoy           || "N/A"],
            ["Depletion",    props.dep_total_str || "N/A"],
            ["Haircut",      props.haircut        || "N/A"],
            ["Adj. NDRP",    props.adj_gva        || "N/A"],
            ["Dep. Sources", props.dep_breakdown  || "N/A"],
        ];
        document.getElementById("panel-stats").innerHTML = rows.map(function(r) {{
            return '<div style="background:#111e33;padding:8px;border-radius:3px;">'
                 + '<div style="color:#7090b0;font-size:8px;text-transform:uppercase;'
                 + 'letter-spacing:.06em;margin-bottom:3px;">' + r[0] + '</div>'
                 + '<div style="color:#f0e8d8;font-size:11px;">' + r[1] + '</div>'
                 + '</div>';
        }}).join("");
        drawSparkline(props.province_key);
        drawSectorBars(props.sectors || {{}});
        drawWaterfall(props.sector_ts || {{}}, period);
        document.getElementById("click-panel").style.display = "block";
    }}

    window.addEventListener("map:ready", function() {{ redraw(currentPeriod); }});
    </script>

    <style>
    .custom-tip {{
        background: #1a2840 !important; color: #f0e8d8 !important;
        font-family: 'Courier New', monospace !important; font-size: 11px !important;
        border: 1px solid #b8860b !important; border-radius: 2px !important;
        padding: 8px !important; line-height: 1.6 !important;
    }}
    </style>

    <!-- Period slider -->
    <div style="position:fixed;bottom:30px;left:50%;transform:translateX(-50%);
                z-index:1000;background:rgba(26,40,64,.95);color:#f0e8d8;
                padding:12px 20px;border:1px solid #b8860b;
                font-family:'Courier New',monospace;font-size:11px;
                box-shadow:0 2px 12px rgba(0,0,0,.5);min-width:340px;text-align:center;">
        <div style="color:#7090b0;font-size:9px;letter-spacing:.08em;
                    text-transform:uppercase;margin-bottom:6px">Period</div>
        <div id="period-label"
             style="font-size:15px;font-weight:700;color:#c9a84c;margin-bottom:8px"></div>
        <input type="range" id="period-slider" min="0"
               max="{_slider_max}" value="{_slider_max}"
               style="width:100%;accent-color:#b8860b;"
               oninput="redraw(ALL_PERIODS[this.value])">
        <div style="display:flex;justify-content:space-between;
                    font-size:8px;color:#506080;margin-top:2px;">
            <span>{_slider_start}</span><span>{_slider_end}</span>
        </div>
    </div>

    <!-- Layer switcher -->
    <div style="position:fixed;top:10px;right:10px;z-index:1000;
                background:rgba(26,40,64,.95);color:#f0e8d8;
                padding:10px;border:1px solid #b8860b;
                font-family:'Courier New',monospace;font-size:10px;">
        <div style="color:#7090b0;font-size:9px;letter-spacing:.08em;
                    text-transform:uppercase;margin-bottom:6px">Layer</div>
        {_layer_buttons}
    </div>

    <!-- Title -->
    <div style="position:fixed;top:10px;left:50px;z-index:1000;
                background:rgba(26,40,64,.92);color:#f0e8d8;
                padding:10px 14px;border-left:3px solid #b8860b;
                font-family:'Courier New',monospace;font-size:11px;
                box-shadow:0 2px 12px rgba(0,0,0,.5);">
        <div style="font-family:sans-serif;font-weight:800;font-size:14px;
                    color:#c9a84c;letter-spacing:.04em">Indonesia GVA Map</div>
        <div style="color:#7090b0;margin-top:2px;font-size:9px;
                    text-transform:uppercase;letter-spacing:.08em">
            Resource-Adjusted Regional Output · ADHK 2010
        </div>
    </div>

    <!-- Click panel -->
    <div id="click-panel"
         style="display:none;position:fixed;top:0;right:0;width:380px;height:100vh;
                background:#0d1a2d;color:#f0e8d8;z-index:2000;overflow-y:auto;
                border-left:2px solid #b8860b;font-family:'Courier New',monospace;
                font-size:11px;box-shadow:-4px 0 20px rgba(0,0,0,.6);">
        <div style="padding:14px 16px;border-bottom:1px solid #1e3050;
                    position:sticky;top:0;background:#0d1a2d;z-index:1;">
            <div style="display:flex;justify-content:space-between;align-items:center;">
                <span id="panel-title"
                      style="font-size:14px;font-weight:700;color:#c9a84c;"></span>
                <button onclick="closePanel()"
                    style="background:none;border:none;color:#7090b0;
                           font-size:18px;cursor:pointer;padding:0 4px;">&#x2715;</button>
            </div>
            <div id="panel-period"
                 style="font-size:9px;color:#7090b0;margin-top:3px;
                        text-transform:uppercase;letter-spacing:.08em;"></div>
        </div>
        <div id="panel-stats"
             style="display:grid;grid-template-columns:1fr 1fr;
                    gap:8px;padding:12px 16px;border-bottom:1px solid #1e3050;"></div>
        <div style="padding:12px 16px;border-bottom:1px solid #1e3050;">
            <div style="font-size:9px;color:#7090b0;text-transform:uppercase;
                        letter-spacing:.08em;margin-bottom:6px;">GVA YoY Growth -- All Periods</div>
            <svg id="sparkline" width="340" height="90" style="overflow:visible;"></svg>
        </div>
        <div style="padding:12px 16px;border-bottom:1px solid #1e3050;">
            <div style="font-size:9px;color:#7090b0;text-transform:uppercase;
                        letter-spacing:.08em;margin-bottom:6px;">Sector Share of GVA (latest period)</div>
            <svg id="sector-bars" width="340" height="160" style="overflow:visible;"></svg>
        </div>
        <div style="padding:12px 16px 24px;">
            <div style="font-size:9px;color:#7090b0;text-transform:uppercase;
                        letter-spacing:.08em;margin-bottom:6px;">Sectoral Contribution to GVA Growth (pp, YoY)</div>
            <svg id="waterfall" width="340" height="240" style="overflow:visible;"></svg>
        </div>
    </div>

    <script>
    document.addEventListener("DOMContentLoaded", function() {{
        setTimeout(function() {{ window.dispatchEvent(new Event("map:ready")); }}, 500);
    }});
    </script>
    """))

    m.save(output)
    print(f"Map saved -> {Path(output).resolve()}")
    return m


# 8.  ENTRY POINT

In [59]:
all_candidate_periods = [
    f"{y}{q}"
    for y in [2021, 2022, 2023, 2024, 2025, 2026]
    for q in ["Q1", "Q2", "Q3", "Q4"]
]
available_periods = set(periods_available) 

valid_years, valid_quarters = set(), set()

for p in all_candidate_periods:
    if p in available_periods:
        valid_years.add(int(p[:4]))
        valid_quarters.add(p[4:])

build_map(
    province_data=PROVINCE_DATA,
    years=sorted(valid_years),
    quarters=["Q1", "Q2", "Q3", "Q4"],
    output="indonesia_gva_depletion_map.html",
)

import webbrowser
webbrowser.open("indonesia_gva_depletion_map.html")

Using GeoJSON: indonesia_provinces.geojson
  Enriched 2021Q1
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2021Q2
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2021Q3
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2021Q4
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2022Q1
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2022Q2
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2022Q3
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2022Q4
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2023Q1
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2023Q2
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2023Q3
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2023Q4
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2024Q1
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2024Q2
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2024Q3
Using GeoJSON: indonesia_provinces.geojson
  Enriched 2024Q4
Using GeoJSON: indonesia

True

# 9.  DEBUG / SANITY CHECK

In [60]:
geo      = load_geojson()
enriched = enrich_geojson(geo, PROVINCE_DATA, 2024, "Q4")
f0       = enriched["features"][0]
print("Province:", f0["properties"].get("province_key"))
for k in ["raw_gva", "yoy", "dep_total_str", "haircut", "adj_gva", "top3_sectors"]:
    print(f"  {k}: {f0['properties'].get(k)}")
print("  sector_ts keys:", list(f0["properties"].get("sector_ts", {}).keys())[:4])

Using GeoJSON: indonesia_provinces.geojson
Province: ACEH
  raw_gva: Rp 40.8T
  yoy: +4.2%
  dep_total_str: Rp2.5T
  haircut: 6.0%
  adj_gva: Rp 38.3T
  top3_sectors: Pertanian: 27% | Perdagangan: 17% | Konstruksi: 9%
  sector_ts keys: ['Pertanian', 'Pertambangan', 'Industri Pengolahan', 'Listrik & Gas']
